# Homework 2: Language Modeling
11-411/611 Natural Language Processing (Spring 2026)

- RELEASED: Thursday, Feb 19th, 2026
- DUE: Thursday, March 12th 2025 11:59 pm EDT

Whether for transcribing spoken utterances as correct word sequences or generating coherent human-like text, language models are extremely useful.

In this assignment, you will be building your own language models powered by n-grams and transformer encoder.

### Submission Guidelines
**Programming:** 
- This notebook contains helpful test cases and additional information about the programming part of the HW. However, you are only required to submit `ngram_lm.py` and `encoder_classifier.py` on Gradescope.
- We recommended that you first code in the notebook and then copy the corresponding methods/classes to `ngram_lm.py` and `encoder_classifier.py`.

**Written:**
- Analysis questions would require you to run your code.
- You need to write your answers in a document and upload it alongside the programming components

### Upload (if using Colab) main.py and utils.py, and the data.zip file

In [ ]:
!unzip data.zip

## Part 1: Language Models [60 points]

### Step 0: Preprocessing

In [2]:
!pip install transformers
!pip install requests
!pip install torch
!pip install tqdm

import math
import torch
import numpy as np
import torch.nn as nn
from collections import Counter

We provide you with a few functions in `utils.py` to read and preprocess your input data. Do not edit this file!

In [3]:
from utils import *

We have performed a round of preprocessing on the datasets.

- Each file contains one sentence per line.
- All punctuation marks have been removed.
- Each line is a sequences of tokens separated by whitespace.

#### Special Symbols ( Already defined in `utils.py` )
The start and end tokens will act as padding to the given sentences, to make sure they are correctly defined, print them here:

In [4]:
print("Sentence START symbol: {}".format(START))
print("Sentence END symbol: {}".format(EOS))
print("Unknown word symbol: {}".format(UNK))

Sentence START symbol: <s>
Sentence END symbol: </s>
Unknown word symbol: <UNK>


#### Reading and processing an example file

In [5]:
# Read the sample file
sample = read_file("data/sample.txt")
print(sample)

['We are never ever ever ever ever getting back together\n', 'We are the ones together we are back']


In [6]:
# Preprocess the content to add corresponding number of start and end tokens. Try out the method with n = 3 and n = 4 as well.
# Preprocessing example for bigrams (n=2)
sample = preprocess(sample, n=3)
for s in sample:
    print(s)

['<s>', '<s>', 'we', 'are', 'never', 'ever', 'ever', 'ever', 'ever', 'getting', 'back', 'together', '</s>']
['<s>', '<s>', 'we', 'are', 'the', 'ones', 'together', 'we', 'are', 'back', '</s>']


In [7]:
# Flattens a nested list into a 1D list.
flattened = flatten(sample)
print(flattened)

['<s>', '<s>', 'we', 'are', 'never', 'ever', 'ever', 'ever', 'ever', 'getting', 'back', 'together', '</s>', '<s>', '<s>', 'we', 'are', 'the', 'ones', 'together', 'we', 'are', 'back', '</s>']


### Step 1: N-Gram Language Model

#### TODO: Defining `get_ngrams()`

In [8]:
#######################################
# TODO: get_ngrams()
#######################################
def get_ngrams(list_of_words, n):
    """
    Returns a list of n-grams for a list of words.
    Args
    ----
    list_of_words: List[str]
        List of already preprocessed and flattened (1D) list of tokens e.g. ["<s>", "hello", "</s>", "<s>", "bye", "</s>"]
    n: int
        n-gram order e.g. 1, 2, 3
    
    Returns:
        n_grams: List[Tuple]
            Returns a list containing n-gram tuples
    """
    n_grams = []
    for i in range(len(list_of_words) - n + 1):
        n_grams.append(tuple(list_of_words[i:i+n]))
    return n_grams


In [9]:
#######################################
# TEST: get_ngrams()
#######################################
sample = preprocess(read_file("data/sample.txt"), n=3)
flattened = flatten(sample)

assert get_ngrams(flattened, 3) == [('<s>', '<s>', 'we'),
        ('<s>', 'we', 'are'),
        ('we', 'are', 'never'),
        ('are', 'never', 'ever'),
        ('never', 'ever', 'ever'),
        ('ever', 'ever', 'ever'),
        ('ever', 'ever', 'ever'),
        ('ever', 'ever', 'getting'),
        ('ever', 'getting', 'back'),
        ('getting', 'back', 'together'),
        ('back', 'together', '</s>'),
        ('together', '</s>', '<s>'),
        ('</s>', '<s>', '<s>'),
        ('<s>', '<s>', 'we'),
        ('<s>', 'we', 'are'),
        ('we', 'are', 'the'),
        ('are', 'the', 'ones'),
        ('the', 'ones', 'together'),
        ('ones', 'together', 'we'),
        ('together', 'we', 'are'),
        ('we', 'are', 'back'),
        ('are', 'back', '</s>')]

#### **TODO:** Class `NGramLanguageModel()`

*Now*, we will define our LanguageModel class.

**Some Useful Variables:**
- self.model: `dict` of n-grams and their corresponding probabilities, keys being the tuple containing the n-gram, and the value being the probability of the n-gram.
- self.vocab: `dict` of unigram vocabulary with counts, keys being the words themselves and the values being their frequency.
- self.n: `int` value for n-gram order (e.g. 1, 2, 3).
- self.train_data: `List[List]` containing preprocessed **unflattened** train sentences. You will have to flatten it to use in the language model
- self.smoothing: `float` flag signifying the smoothing parameter.

In `lm.py`, we will be taking most of these argumemts from the command line using this command:

`python3 lm.py --train data/sample.txt --test data/sample.txt --n 3 --smoothing 0 --min_freq 1`

Note that we will not be using log probabilities in this section. Store the probabilities as they are, not in log space.

**Laplace Smoothing**

There are two ways to perform this:
- Either you calculate all possible n-grams at train time and calculate smooth probabilities for all of them, hence inflating the model (eager emoothing). You then use the probabilities as when required at test time. **OR**
- You calculate the probabilities for the **observed n-grams** at train time, using the smoothed likelihood formula, then if any unseen n-gram is observed at test time, you calculate the probability using the smoothed likelihood formula and store it in the model for future use (lazy smoothing).

You will be implementing lazy smoothing

**Perplexity**

Steps:
1. Flatten the test data.
2. Extract ngrams from the flattened data.
3. Calculate perplexity according to given formula. For unseen n-grams, calculate using smoothed likelihood and store the unseen n-gram probability in the labguage model `model` attribute:

$ppl(W_{test}) = ppl(W_1W_2 ... W_n)^{-1/n} $

Tips:
- Remember that product changes to summation under `log`. Take the log of probabilities, sum them up, and then exponentiate it to get back to the original scale.
- Make sure to `flatten()` your data before creating the n_grams using `get_ngrams()`.
- The test suite provided is **not exhaustive**.


In [23]:
#######################################
# TODO: NGramLanguageModel()
#######################################
class NGramLanguageModel():
    def __init__(self, n, train_data, alpha=1):
        """
        Language model class.

        Args
        ____
        n: int
            n-gram order
        train_data: List[List]
            already preprocessed unflattened list of sentences. e.g. [["<s>", "hello", "my", "</s>"], ["<s>", "hi", "there", "</s>"]]
        alpha: float
            Smoothing parameter

        Other attributes:
            self.tokens: list of individual tokens present in the training corpus
            self.vocab: vocabulary dict with counts
            self.model: n-gram language model, i.e., n-gram dict with probabilties
            self.n_grams_counts: dictionary for storing the frequency of ngrams in the training data, keys being the tuple of words(n-grams) and value being their frequency
            self.prefix_counts: dictionary for storing the frequency of the (n-1) grams in the data, similar to the self.n_grams_counts
            As an example:
            For a trigram model, the n-gram would be (w1,w2,w3), the corresponding [n-1] gram would be (w1,w2)
        """
        self.n = n
        self.train_data = train_data
        self.smoothing = alpha
        self.tokens = []
        self.vocab = {}
        self.model = {}
        self.n_grams_counts = {}
        self.prefix_counts = {}

        self.build()


    def build(self):
        """
        Returns a n-gram dict with their smoothed probabilities. Remember to consider the edge case of n=1 as well

        You are expected to update the self.n_grams_counts and self.prefix_counts, and use those calculate the probabilities.
        """
        flattened_train_data = flatten(self.train_data)
        self.tokens = flattened_train_data
        self.vocab = Counter(flattened_train_data)
        n_grams = get_ngrams(flattened_train_data, self.n)
        self.n_grams_counts = Counter(n_grams)
        # Calculate prefix counts for higher-order n-grams
        if self.n > 1:
            for ngram in n_grams:
                prefix = ngram[:-1]
                self.prefix_counts[prefix] = self.prefix_counts.get(prefix, 0) + 1

        probs = self.get_smooth_probabilities(self.n_grams_counts)
        self.model = probs

        return probs


    def get_smooth_probabilities(self, ngrams):
        """
        Returns the smoothed probability of the n-gram, using Laplace Smoothing.
        Remember to consider the edge case of  n = 1
        HINT: Use self.n_gram_counts, self.tokens and self.prefix_counts
        """
        probs = {}
        V = len(self.vocab)  # Vocabulary size

        if self.n == 1:
            # For unigrams, use the frequency counts directly
            total_tokens = len(self.tokens)
            for ngram, count in ngrams.items():
                probs[ngram] = (count + self.smoothing) / (total_tokens + self.smoothing * V)
        else:
            # For higher-order n-grams, use the prefix counts
            for ngram, count in ngrams.items():
                prefix = ngram[:-1]
                prefix_count = self.prefix_counts.get(prefix, 0)
                probs[ngram] = (count + self.smoothing) / (prefix_count + self.smoothing * V)

        return probs

    def get_prob(self, ngram):
        """
        Returns the probability of the n-gram, using Laplace Smoothing.

        Args
        ____
        ngram: tuple
            n-gram tuple

        Returns
        _______
        float
            probability of the n-gram
        """
        if ngram in self.model:
            return self.model[ngram]
        
        V =  len(self.vocab)  # Vocabulary size
        if self.n == 1:
            total_tokens = len(self.tokens)
            prob = self.smoothing / (total_tokens + self.smoothing * V)
        else:
            prefix = ngram[:-1]
            prefix_count = self.prefix_counts.get(prefix, 0)
            prob = self.smoothing / (prefix_count + self.smoothing * V)
        
        self.model[ngram] = prob
        
        return prob

    def perplexity(self, test_data):
        """
        Returns perplexity calculated on the test data.
        Args
        ----------
        test_data: List[List]
            Already preprocessed nested list of sentences

        Returns
        -------
        float
            Calculated perplexity value
        """
        total_log_prob = 0

        flattened_test_data = flatten(test_data)
        N_tokens = len(flattened_test_data)
        if N_tokens == 0:
            return 0.0
        
        ngrams = get_ngrams(flattened_test_data, self.n)

        for ngram in ngrams:
            prob = self.get_prob(ngram)
            if prob == 0:
                return float('inf')
            total_log_prob += math.log(prob)
        perplexity = math.exp(-total_log_prob / N_tokens)
        return perplexity


In [19]:
#######################################
# TEST: NGramLanguageModel()
#######################################
# For the sake of understanding we will pass alpha as 0 (no smoothing), so that you gain intuition about the probabilities
sample = preprocess(read_file("data/sample.txt"), n=2)
test_lm = NGramLanguageModel(n=2, train_data=sample, alpha=0)

expected_vocab = Counter({'<s>': 2,
        'we': 3,
        'are': 3,
        'never': 1,
        'ever': 4,
        'getting': 1,
        'back': 2,
        'together': 2,
        '</s>': 2,
        'the': 1,
        'ones': 1})

expected_model = {('<s>', 'we'): 1.0,
        ('we', 'are'): 1.0,
        ('are', 'never'): 0.3333333333333333,
        ('never', 'ever'): 1.0,
        ('ever', 'ever'): 0.75,
        ('ever', 'getting'): 0.25,
        ('getting', 'back'): 1.0,
        ('back', 'together'): 0.5,
        ('together', '</s>'): 0.5,
        ('</s>', '<s>'): 1.0,
        ('are', 'the'): 0.3333333333333333,
        ('the', 'ones'): 1.0,
        ('ones', 'together'): 1.0,
        ('together', 'we'): 0.5,
        ('are', 'back'): 0.3333333333333333,
        ('back', '</s>'): 0.5}

assert test_lm.vocab == expected_vocab, f"Vocabulary mismatch! Expected: {expected_vocab}, but got: {test_lm.vocab}"

assert test_lm.model == expected_model, (
    f"Model mismatch! \n"
    f"Expected keys but missing: {set(expected_model.keys()) - set(test_lm.model.keys())}\n"
    f"Unexpected keys in model: {set(test_lm.model.keys()) - set(expected_model.keys())}\n"
    f"Discrepancies in probabilities: "
    f"{ {k: (expected_model[k], test_lm.model[k]) for k in expected_model if k in test_lm.model and expected_model[k] != test_lm.model[k]} }"
)

In [40]:
# business_data = preprocess("data/bbc/business.txt")
# sports_data = preprocess("data/bbc/sports.txt")

print("--- Business Dataset ---")
# Q1: Business 2-grams
business_data = preprocess("data/bbc/business.txt", n=2)
biz_model_2 = NGramLanguageModel(n=2, train_data=business_data)
biz_unique_2grams = len(biz_model_2.n_grams_counts)
print(f"1. Unique 2-grams in Business: {biz_unique_2grams}")

# Q2: Business 3-grams
business_data = preprocess("data/bbc/business.txt", n=3)
biz_model_3 = NGramLanguageModel(n=3, train_data=business_data)
biz_unique_3grams = len(biz_model_3.n_grams_counts)
print(f"2. Unique 3-grams in Business: {biz_unique_3grams}")

print("\n--- Sports Dataset ---")
# Q3: Sports 2-grams
sports_data = preprocess("data/bbc/sports.txt", n=2)
sports_model_2 = NGramLanguageModel(n=2, train_data=sports_data)
sports_unique_2grams = len(sports_model_2.n_grams_counts)
print(f"3. Unique 2-grams in Sports: {sports_unique_2grams}")

# Q4: Sports 3-grams
sports_data = preprocess("data/bbc/sports.txt", n=3)
sports_model_3 = NGramLanguageModel(n=3, train_data=sports_data)
sports_unique_3grams = len(sports_model_3.n_grams_counts)
print(f"4. Unique 3-grams in Sports: {sports_unique_3grams}")

--- Business Dataset ---
1. Unique 2-grams in Business: 27
2. Unique 3-grams in Business: 40

--- Sports Dataset ---
3. Unique 2-grams in Sports: 25
4. Unique 3-grams in Sports: 37


In [41]:
# 计算 Business 数据集的理论最大值
V = len(biz_model_2.vocab) # 获取词汇表大小 |V|
print(f"Vocabulary Size (|V|): {V}")
print(f"Possible 2-grams (|V|^2): {V ** 2}")
print(f"Possible 3-grams (|V|^3): {V ** 3}")

Vocabulary Size (|V|): 15
Possible 2-grams (|V|^2): 225
Possible 3-grams (|V|^3): 3375


In [20]:
#######################################
# TEST smoothing: NGramLanguageModel()
#######################################
sample = preprocess(read_file("data/sample.txt"), n=2)
test_lm = NGramLanguageModel(n=2, train_data=sample, alpha=1)

expected_vocab_smoothing = Counter({'<s>': 2,
        'we': 3,
        'are': 3,
        'never': 1,
        'ever': 4,
        'getting': 1,
        'back': 2,
        'together': 2,
        '</s>': 2,
        'the': 1,
        'ones': 1})

expected_model_smoothing ={('<s>', 'we'): 0.23076923076923078,
        ('we', 'are'): 0.2857142857142857,
        ('are', 'never'): 0.14285714285714285,
        ('never', 'ever'): 0.16666666666666666,
        ('ever', 'ever'): 0.26666666666666666,
        ('ever', 'getting'): 0.13333333333333333,
        ('getting', 'back'): 0.16666666666666666,
        ('back', 'together'): 0.15384615384615385,
        ('together', '</s>'): 0.15384615384615385,
        ('</s>', '<s>'): 0.16666666666666666,
        ('are', 'the'): 0.14285714285714285,
        ('the', 'ones'): 0.16666666666666666,
        ('ones', 'together'): 0.16666666666666666,
        ('together', 'we'): 0.15384615384615385,
        ('are', 'back'): 0.14285714285714285,
        ('back', '</s>'): 0.15384615384615385}


assert test_lm.vocab == expected_vocab_smoothing, f"Vocabulary mismatch! Expected: {expected_vocab}, but got: {test_lm.vocab}"

assert test_lm.model == expected_model_smoothing, (
    f"Model mismatch! \n"
    f"Expected keys but missing: {set(expected_model_smoothing.keys()) - set(test_lm.model.keys())}\n"
    f"Unexpected keys in model: {set(test_lm.model.keys()) - set(expected_model_smoothing.keys())}\n"
    f"Discrepancies in probabilities: "
    f"{ {k: (expected_model_smoothing[k], test_lm.model[k]) for k in expected_model_smoothing if k in test_lm.model and expected_model_smoothing[k] != test_lm.model[k]} }"
)

In [21]:
#######################################
# TEST unigram: NGramLanguageModel()
#######################################
sample = preprocess(read_file("data/sample.txt"), n=1)
test_lm = NGramLanguageModel(n=1, train_data=sample, alpha=1)

expected_vocab_unigram = Counter({'<s>': 2,
        'we': 3,
        'are': 3,
        'never': 1,
        'ever': 4,
        'getting': 1,
        'back': 2,
        'together': 2,
        '</s>': 2,
        'the': 1,
        'ones': 1})

expected_model_unigram = {('<s>',): 0.09090909090909091,
        ('we',): 0.12121212121212122,
        ('are',): 0.12121212121212122,
        ('never',): 0.06060606060606061,
        ('ever',): 0.15151515151515152,
        ('getting',): 0.06060606060606061,
        ('back',): 0.09090909090909091,
        ('together',): 0.09090909090909091,
        ('</s>',): 0.09090909090909091,
        ('the',): 0.06060606060606061,
        ('ones',): 0.06060606060606061}


assert test_lm.vocab == expected_vocab_unigram, f"Vocabulary mismatch! Expected: {expected_vocab}, but got: {test_lm.vocab}"

assert test_lm.model == expected_model_unigram, (
    f"Model mismatch! \n"
    f"Expected keys but missing: {set(expected_model_unigram.keys()) - set(test_lm.model.keys())}\n"
    f"Unexpected keys in model: {set(test_lm.model.keys()) - set(expected_model_unigram.keys())}\n"
    f"Discrepancies in probabilities: "
    f"{ {k: (expected_model_unigram[k], test_lm.model[k]) for k in expected_model_unigram if k in test_lm.model and expected_model_unigram[k] != test_lm.model[k]} }"
)

In [24]:
#######################################
# TEST: perplexity()
#######################################
test_lm = NGramLanguageModel(n=2, train_data=sample, alpha=0)
test_ppl = test_lm.perplexity(sample)
assert test_ppl < 1.7
assert test_ppl > 0

test_lm = NGramLanguageModel(n=2, train_data=sample, alpha=1)
test_ppl = test_lm.perplexity(sample)
assert test_ppl < 5.0
assert test_ppl > 0

### Step 2: Transformer Encoder for Record Classification

Transformer encoders (like BERT) have revolutionized NLP by using self-attention mechanisms to capture contextual relationships between words. Unlike RNNs which process sequences sequentially, transformers can attend to all positions in a sequence simultaneously, making them highly effective for understanding context.

In this section, you will build a small transformer encoder from scratch for the task of **record classification** - given a song lyric, predict which record (album) it belongs to. We will classify lyrics into three records:
- Record 1 (s1): "The Tortured Poets Department"
- Record 2 (s2): "So Long, London"  
- Record 3 (s3): "Down Bad"

Key Concepts:
- Self-Attention: Allows the model to weigh the importance of different words when encoding each word
- Multi-Head Attention: Multiple attention heads capture different types of relationships
- Positional Encoding: Since transformers don't have inherent sequence ordering, we add positional information
- Classification Head: A linear layer on top of the encoder for classification

What we provide:
- Pre-trained GloVe word embeddings (50-dimensional)
- Data loading and preprocessing utilities
- Training loop and evaluation code

What you need to implement:
- `PositionalEncoding` class: Add positional information to embeddings
- `TransformerEncoderClassifier` class: The main encoder model for classification

Before diving into building Transformer Encoders using PyTorch, it's essential to have a solid foundation in the following areas:
We assume you have a basic understanding of PyTorch and its core concepts, including tensors, autograd, modules (nn.Module), and how to construct simple neural networks using PyTorch. For more comprehensive learning, refer to the [PyTorch official tutorials](https://pytorch.org/tutorials/) and documentation. Also refer to the original ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) paper.

#### Preparing the Data for Record Classification

For our record classification task, we need to prepare a dataset where each sample is a lyric line paired with the record label. We have lyrics from three different records in `data/lyrics/`.

The following code sets up:
1. Loading lyrics data for each record
2. Building vocabulary and loading pre-trained GloVe embeddings
3. Creating the embedding matrix (provided to you)
4. A custom Dataset class for loading lyrics with labels
5. DataLoaders for training and evaluation

In [25]:
# Load and prepare the classification dataset
from pathlib import Path
import os
import random

# Define the three records we're classifying
RECORDS = {
    "s1": "the_tortured_poets_department",
    "s2": "so_long_london", 
    "s3": "down_bad"
}

def load_record_classification_data(data_dir="data/lyrics/", max_seq_len=64, train_ratio=0.8):
    """
    Load lyrics data for record classification.
    
    Returns:
        train_data: List of (tokens, record_label) for training
        test_data: List of (tokens, record_label) for testing
        vocab: Set of unique words
        word_to_ix: Dict mapping words to indices
        ix_to_word: Dict mapping indices to words
        record_to_ix: Dict mapping record names to class indices
        ix_to_record: Dict mapping class indices to record names
        max_seq_len: Maximum sequence length
    """
    all_lyrics = []  # List of (lyric_tokens, record_name)
    all_tokens = []
    
    # Load lyrics for each record
    for record_label, record_file in RECORDS.items():
        filepath = Path(data_dir) / f"{record_file}.txt"
        if filepath.exists():
            lines = read_file(filepath)
            for line in lines:
                tokens = line.strip().lower().split()
                if len(tokens) > 2:  # Filter out very short lines
                    all_lyrics.append((tokens, record_label))
                    all_tokens.extend(tokens)
    
    # Build vocabulary
    vocab = set(all_tokens)
    vocab.add(START)
    vocab.add(EOS)
    vocab.add(UNK)
    vocab.add("<PAD>")  # Padding token for batching
    
    word_to_ix = {word: i for i, word in enumerate(sorted(vocab))}
    ix_to_word = {i: word for word, i in word_to_ix.items()}
    
    # Build record label mapping
    record_to_ix = {record: i for i, record in enumerate(sorted(RECORDS.keys()))}
    ix_to_record = {i: record for record, i in record_to_ix.items()}
    
    # Shuffle and split data
    random.seed(11411)
    random.shuffle(all_lyrics)
    split_idx = int(len(all_lyrics) * train_ratio)
    train_data = all_lyrics[:split_idx]
    test_data = all_lyrics[split_idx:]
    
    print(f"Total samples: {len(all_lyrics)}")
    print(f"Training samples: {len(train_data)}")
    print(f"Test samples: {len(test_data)}")
    print(f"Vocabulary size: {len(vocab)}")
    print(f"Number of records (classes): {len(RECORDS)}")
    print(f"Records: {list(RECORDS.keys())}")
    
    return train_data, test_data, vocab, word_to_ix, ix_to_word, record_to_ix, ix_to_record, max_seq_len

# Load the data
train_data, test_data, vocab, word_to_ix, ix_to_word, record_to_ix, ix_to_record, max_seq_len = load_record_classification_data()

Total samples: 171
Training samples: 136
Test samples: 35
Vocabulary size: 411
Number of records (classes): 3
Records: ['s1', 's2', 's3']


#### Loading Pre-trained GloVe Embeddings

We provide pre-trained GloVe embeddings (50-dimensional) for you to use. These embeddings capture semantic relationships between words learned from a large corpus.

The `get_embedding_matrix()` function creates an embedding matrix where:
- Words found in GloVe use their pre-trained vectors
- Words not in GloVe (including special tokens) are randomly initialized

You will use this embedding matrix directly in your model - you don't need to implement an embedding layer from scratch.

In [26]:
def load_glove_embeddings(path):
    """
    Load GloVe embeddings from file.
    
    Args:
        path: Path to the GloVe file (e.g., 'glove.6B.50d.txt')
    
    Returns:
        embeddings_dict: Dict mapping words to their embedding vectors
    """
    embeddings_dict = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = torch.tensor([float(val) for val in values[1:]], dtype=torch.float)
            embeddings_dict[word] = vector
    return embeddings_dict

def get_embedding_matrix(word_to_ix, glove_embeddings, embedding_dim=50):
    """
    Create an embedding matrix using GloVe embeddings.
    
    Args:
        word_to_ix: Dict mapping words to indices
        glove_embeddings: Dict mapping words to GloVe vectors
        embedding_dim: Dimension of embeddings (default 50 for glove.6B.50d)
    
    Returns:
        embedding_matrix: torch.Tensor of shape (vocab_size, embedding_dim)
    """
    vocab_size = len(word_to_ix)
    embedding_matrix = torch.zeros((vocab_size, embedding_dim))
    
    for word, idx in word_to_ix.items():
        if word in glove_embeddings:
            embedding_matrix[idx] = glove_embeddings[word]
        else:
            # Random initialization for words not in GloVe (including special tokens)
            embedding_matrix[idx] = torch.randn(embedding_dim) * 0.1
    
    return embedding_matrix

# Load GloVe embeddings
glove_path = 'glove.6B.50d.txt'
glove_embeddings = load_glove_embeddings(glove_path)
print(f"Loaded {len(glove_embeddings)} GloVe embeddings")

# Create embedding matrix - THIS IS PROVIDED TO YOU
EMBEDDING_DIM = 50
embedding_matrix = get_embedding_matrix(word_to_ix, glove_embeddings, EMBEDDING_DIM)
print(f"Embedding matrix shape: {embedding_matrix.shape}")

Loaded 400000 GloVe embeddings
Embedding matrix shape: torch.Size([411, 50])


#### Dataset Class for Record Classification

The following Dataset class handles:
- Converting tokens to indices using the vocabulary
- Padding/truncating sequences to a fixed length
- Creating attention masks to distinguish real tokens from padding

In [27]:
# Dataset class for record classification
class RecordClassificationDataset(Dataset):
    def __init__(self, data, word_to_ix, record_to_ix, max_seq_len):
        """
        Args:
            data: List of (tokens, record_label) tuples
            word_to_ix: Dict mapping words to indices
            record_to_ix: Dict mapping record labels to indices
            max_seq_len: Maximum sequence length (for padding/truncation)
        """
        self.data = data
        self.word_to_ix = word_to_ix
        self.record_to_ix = record_to_ix
        self.max_seq_len = max_seq_len
        self.pad_idx = word_to_ix["<PAD>"]
        self.unk_idx = word_to_ix[UNK]
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        tokens, record = self.data[idx]
        
        # Convert tokens to indices
        indices = [self.word_to_ix.get(token, self.unk_idx) for token in tokens]
        
        # Truncate or pad to max_seq_len
        if len(indices) > self.max_seq_len:
            indices = indices[:self.max_seq_len]
        else:
            indices = indices + [self.pad_idx] * (self.max_seq_len - len(indices))
        
        # Create attention mask (1 for real tokens, 0 for padding)
        attention_mask = [1 if idx != self.pad_idx else 0 for idx in indices]
        
        return (
            torch.tensor(indices, dtype=torch.long),
            torch.tensor(attention_mask, dtype=torch.long),
            torch.tensor(self.record_to_ix[record], dtype=torch.long)
        )

# Create datasets and dataloaders
train_dataset = RecordClassificationDataset(train_data, word_to_ix, record_to_ix, max_seq_len)
test_dataset = RecordClassificationDataset(test_data, word_to_ix, record_to_ix, max_seq_len)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Number of training batches: {len(train_dataloader)}")
print(f"Number of test batches: {len(test_dataloader)}")

Number of training batches: 5
Number of test batches: 2


#### TODO: Implementing the Transformer Encoder Classifier

You will implement two classes:

1. `PositionalEncoding`: Adds positional information to the word embeddings since transformers have no inherent notion of position.

2. `TransformerEncoderClassifier`: The main model that:
   - Uses the provided GloVe embedding matrix
   - Adds positional encoding
   - Passes through transformer encoder layers
   - Classifies into one of three records

In [34]:
#######################################
# TODO: PositionalEncoding()
#######################################
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        """
        Positional Encoding module that adds positional information to embeddings.

        Args
        ____
        d_model: int
            Dimension of the model (embedding dimension, should be 50 for GloVe)
        max_len: int
            Maximum sequence length
        dropout: float
            Dropout rate

        HINT: Use sine and cosine functions of different frequencies.
        For position pos and dimension i:
            PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
            PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

        Store the positional encodings as a buffer (not a parameter) using:
            self.register_buffer('pe', pe_tensor)
        """
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape (1, max_len, d_model)
        self.register_buffer('pe', pe)


    def forward(self, x):
        """
        Add positional encoding to input embeddings.

        Args
        ____
        x: torch.Tensor
            Input tensor of shape (batch_size, sequence_length, d_model)

        Returns
        -------
        torch.Tensor
            Output tensor with positional encoding added, same shape as input
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


#######################################
# TODO: TransformerEncoderClassifier()
#######################################
class TransformerEncoderClassifier(nn.Module):
    def __init__(self, embedding_matrix, num_classes, nhead=2, num_layers=2,
                 dim_feedforward=128, dropout=0.1, max_seq_len=64):
        """
        Transformer Encoder for record classification.

        Args
        ____
        embedding_matrix: torch.Tensor
            Pre-trained embedding matrix of shape (vocab_size, d_model)
            Use this directly - do NOT create a new nn.Embedding from scratch
        num_classes: int
            Number of output classes (3 for our records: s1, s2, s3)
        nhead: int
            Number of attention heads (d_model must be divisible by nhead)
        num_layers: int
            Number of transformer encoder layers
        dim_feedforward: int
            Dimension of the feedforward network in each encoder layer
        dropout: float
            Dropout rate
        max_seq_len: int
            Maximum sequence length

        Attributes to create:
            self.embedding: nn.Embedding
                Initialize from embedding_matrix using nn.Embedding.from_pretrained()
                Set freeze=False to allow fine-tuning
            self.d_model: int
                Embedding dimension (get from embedding_matrix.shape[1])
            self.positional_encoding: PositionalEncoding
                Your positional encoding module
            self.transformer_encoder: nn.TransformerEncoder
                Stack of transformer encoder layers
                Use nn.TransformerEncoderLayer and nn.TransformerEncoder
            self.fc: nn.Linear
                Classification head (d_model -> num_classes)
            self.dropout: nn.Dropout
                Dropout layer

        Note: You may use nn.TransformerEncoderLayer and nn.TransformerEncoder from PyTorch
        """
        super(TransformerEncoderClassifier, self).__init__()

        self.device = torch.device("mps" if torch.backends.mps.is_available()
                                 else "cuda" if torch.cuda.is_available()
                                 else "cpu")
        print(f"Using device: {self.device}")

        # TODO: Your code here
        # 1. Store d_model from embedding_matrix shape
        self.d_model = embedding_matrix.shape[1]
        # 2. Create embedding layer from pre-trained embedding_matrix
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)
        # 3. Initialize positional encoding
        self.positional_encoding = PositionalEncoding(d_model=self.d_model, max_len=max_seq_len, dropout=dropout)
        # 4. Initialize transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(d_model=self.d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=False)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        # 5. Initialize classification head (fc layer)
        self.fc = nn.Linear(self.d_model, num_classes)
        # 6. Initialize dropout
        self.dropout = nn.Dropout(dropout)

        # End of your code
        self.to(self.device)



    def forward(self, x, attention_mask=None):
        """
        Forward pass of the transformer encoder classifier.

        Args
        ____
        x: torch.Tensor
            Input tensor of shape (batch_size, sequence_length) containing token indices
        attention_mask: torch.Tensor
            Attention mask of shape (batch_size, sequence_length)
            1 for real tokens, 0 for padding tokens

        Returns
        -------
        logits: torch.Tensor
            Output logits of shape (batch_size, num_classes)

        HINTS:
        1. Get embeddings from self.embedding(x)
        2. Scale embeddings by sqrt(d_model) (helps with training stability)
        3. Add positional encoding
        4. Transpose for transformer: (batch, seq, dim) -> (seq, batch, dim)
        5. Create key_padding_mask for transformer: True where padding, False where real tokens
           (This is the OPPOSITE of attention_mask!)
        6. Pass through transformer encoder
        7. Transpose back: (seq, batch, dim) -> (batch, seq, dim)
        8. Pool the sequence (mean pooling over non-padded tokens recommended)
        9. Pass through classification head

        Note: Don't forget to move tensors to the correct device!
        """
        x = x.to(self.device)
        embedded = self.embedding(x)
        embedded = embedded * math.sqrt(self.d_model)
        embedded = self.positional_encoding(embedded)
        embedded = embedded.transpose(0, 1)  # (batch, seq, dim) -> (seq, batch, dim)

        if attention_mask is not None:
            attention_mask = attention_mask.to(self.device)
            key_padding_mask = (attention_mask == 0).bool()  # Boolean mask
        else:
            key_padding_mask = None

        encoded = self.transformer_encoder(embedded, src_key_padding_mask=key_padding_mask)
        encoded = encoded.transpose(0, 1)  # (seq, batch, dim) -> (batch, seq, dim)
        # Mean pooling over non-padded tokens
        if attention_mask is not None:
            # Expand attention mask to match the embedding dimension: (batch_size, seq_len, d_model)
            mask_expanded = attention_mask.unsqueeze(-1).expand_as(encoded).float()
            
            # Sum the embeddings for real tokens
            sum_embeddings = torch.sum(encoded * mask_expanded, dim=1)
            
            # Count the number of real tokens (clamp to 1e-9 to avoid division by zero)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            
            # Mean pooling
            pooled = sum_embeddings / sum_mask
        else:
            # Fallback if no mask is provided
            pooled = encoded.mean(dim=1)

        pooled = self.dropout(pooled)
        logits = self.fc(pooled)

        return logits


    def predict(self, x, attention_mask=None):
        """
        Predict class labels for input sequences.

        Args
        ____
        x: torch.Tensor
            Input tensor of shape (batch_size, sequence_length)
        attention_mask: torch.Tensor
            Attention mask of shape (batch_size, sequence_length)

        Returns
        -------
        predictions: torch.Tensor
            Predicted class indices of shape (batch_size,)
        """
        self.eval()
        with torch.no_grad():
            logits = self.forward(x, attention_mask)
            predictions = torch.argmax(logits, dim=1)
        return predictions

In [29]:
#######################################
# TEST: PositionalEncoding()
#######################################
# Test that positional encoding has correct shape and properties
test_pe = PositionalEncoding(d_model=50, max_len=100, dropout=0.0)

# Test input
test_input = torch.randn(2, 10, 50)  # (batch_size=2, seq_len=10, d_model=50)
test_output = test_pe(test_input)

assert test_output.shape == test_input.shape, f"Shape mismatch: expected {test_input.shape}, got {test_output.shape}"

# Check that different positions have different encodings
pe_values = test_pe.pe[0, :5, :5]  # First 5 positions, first 5 dimensions
print("Positional encoding values (first 5 positions, first 5 dims):")
print(pe_values)

# Verify that position 0 and position 1 are different
assert not torch.allclose(test_pe.pe[0, 0], test_pe.pe[0, 1]), "Position 0 and 1 should have different encodings"

Positional encoding values (first 5 positions, first 5 dims):
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000],
        [ 0.8415,  0.5403,  0.6379,  0.7701,  0.4606],
        [ 0.9093, -0.4161,  0.9825,  0.1860,  0.8176],
        [ 0.1411, -0.9900,  0.8753, -0.4835,  0.9909],
        [-0.7568, -0.6536,  0.3656, -0.9308,  0.9415]])


#### Training the Transformer Encoder Classifier

The following code trains the transformer encoder classifier for record classification. The model learns to predict which record (s1, s2, or s3) a given lyric line belongs to.

In [35]:
#######################################
# TEST: TransformerEncoderClassifier() and training
#######################################
torch.manual_seed(11411)

# Hyperparameters
num_classes = len(record_to_ix)  # 3 classes: s1, s2, s3
nhead = 2  # Note: EMBEDDING_DIM (50) must be divisible by nhead
num_layers = 2
dim_feedforward = 128
dropout = 0.1
num_epochs = 10
learning_rate = 0.001

# Initialize the model with the pre-trained embedding matrix
model = TransformerEncoderClassifier(
    embedding_matrix=embedding_matrix,
    num_classes=num_classes,
    nhead=nhead,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    max_seq_len=max_seq_len
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, attention_mask, labels) in enumerate(train_dataloader):
        inputs = inputs.to(model.device)
        attention_mask = attention_mask.to(model.device)
        labels = labels.to(model.device)
        
        optimizer.zero_grad()
        outputs = model(inputs, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_acc = 100 * correct / total
    avg_loss = total_loss / len(train_dataloader)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, Train Accuracy: {train_acc:.2f}%')

Using device: cpu
Epoch 1/10, Loss: 1.1015, Train Accuracy: 37.50%


C:\Users\17293\AppData\Local\Temp\ipykernel_13784\460695386.py:117: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Epoch 2/10, Loss: 1.0093, Train Accuracy: 49.26%
Epoch 3/10, Loss: 0.9376, Train Accuracy: 61.76%
Epoch 4/10, Loss: 0.8478, Train Accuracy: 61.76%
Epoch 5/10, Loss: 0.7322, Train Accuracy: 73.53%
Epoch 6/10, Loss: 0.6151, Train Accuracy: 77.21%
Epoch 7/10, Loss: 0.5832, Train Accuracy: 77.94%
Epoch 8/10, Loss: 0.5180, Train Accuracy: 77.94%
Epoch 9/10, Loss: 0.4560, Train Accuracy: 82.35%
Epoch 10/10, Loss: 0.3721, Train Accuracy: 85.29%


In [36]:
#######################################
# TEST: Evaluation on Test Set
#######################################
def evaluate_model(model, test_dataloader):
    """
    Evaluate the model on the test set.
    
    Args:
        model: The trained TransformerEncoderClassifier
        test_dataloader: DataLoader for test data
    
    Returns:
        accuracy: Test accuracy as a percentage
    """
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, attention_mask, labels in test_dataloader:
            inputs = inputs.to(model.device)
            attention_mask = attention_mask.to(model.device)
            labels = labels.to(model.device)
            
            predictions = model.predict(inputs, attention_mask)
            total += labels.size(0)
            correct += (predictions == labels).sum().item()
    
    accuracy = 100 * correct / total
    return accuracy

test_accuracy = evaluate_model(model, test_dataloader)
print(f"Test Accuracy: {test_accuracy:.2f}%")

# Assert that the model achieves reasonable accuracy (better than random ~33%)
assert test_accuracy > 40, f"Model accuracy {test_accuracy:.2f}% is too low. Expected > 40% (random would be ~33.3%)"

Test Accuracy: 42.86%


#### Evaluating the Transformer Encoder Classifier

The following code evaluates your trained model on the test set. Your model should achieve significantly better than random accuracy (which would be ~33% for 3 classes).

## Part 2: Written [40 points]. We have given some code for some of the written parts to make it easier for you.

### **Written 4.2** – Song Attribution [8 points]

The cell below contains helper code to create an n-gram model for song attribution. You can use this code to train a model on the file provided in the train variable and compute the model's perplexity on the test lyrics. 

In [45]:
# Example code for Taylor Swift N-Gram LM
n = 2
smoothing = 0.1
min_freq = 1

train = read_file("data/lyrics/taylor_swift.txt")
test = read_file("data/lyrics/test_lyrics.txt")

train = preprocess(train, n)
test = preprocess(test, n)
lm = NGramLanguageModel(n, train, smoothing)

ppl = lm.perplexity(test)
print(ppl)

82.86766146846585


In [42]:
# 要测试的三个歌手的文件名
artists = ["taylor_swift.txt", "green_day.txt", "ed_sheeran.txt"]
test_text = read_file("data/lyrics/test_lyrics.txt")

smoothing = 0.1

print("=== Q1: Tri-gram (n=3) Perplexities ===")
test_processed_3 = preprocess(test_text, 3)
ppl_3_dict = {}

for artist in artists:
    train_text = read_file(f"data/lyrics/{artist}")
    train_processed_3 = preprocess(train_text, 3)
    # 训练 3-gram 模型
    lm_3 = NGramLanguageModel(3, train_processed_3, smoothing)
    ppl = lm_3.perplexity(test_processed_3)
    ppl_3_dict[artist] = ppl
    print(f"{artist}: {ppl:.2f}")

# 找出 3-gram 困惑度最低的歌手
likely_artist_3 = min(ppl_3_dict, key=ppl_3_dict.get)
print(f"\n=> Q2 Answer: According to tri-gram, the lyricist is most likely: {likely_artist_3}")

print("\n--------------------------------------------------\n")

print("=== Q3: Bi-gram (n=2) Perplexities ===")
test_processed_2 = preprocess(test_text, 2)
ppl_2_dict = {}

for artist in artists:
    train_text = read_file(f"data/lyrics/{artist}")
    train_processed_2 = preprocess(train_text, 2)
    # 训练 2-gram 模型
    lm_2 = NGramLanguageModel(2, train_processed_2, smoothing)
    ppl = lm_2.perplexity(test_processed_2)
    ppl_2_dict[artist] = ppl
    print(f"{artist}: {ppl:.2f}")

# 找出 2-gram 困惑度最低的歌手
likely_artist_2 = min(ppl_2_dict, key=ppl_2_dict.get)
print(f"\n=> Q3 Answer: According to bi-gram, the lyricist is most likely: {likely_artist_2}")

=== Q1: Tri-gram (n=3) Perplexities ===
taylor_swift.txt: 116.10
green_day.txt: 419.51
ed_sheeran.txt: 418.52

=> Q2 Answer: According to tri-gram, the lyricist is most likely: taylor_swift.txt

--------------------------------------------------

=== Q3: Bi-gram (n=2) Perplexities ===
taylor_swift.txt: 82.87
green_day.txt: 256.87
ed_sheeran.txt: 267.38

=> Q3 Answer: According to bi-gram, the lyricist is most likely: taylor_swift.txt


Below, contains helper code for questions 4.2.4 and 4.2.5. We use a helper function split_and_save_datasets, to save the train and test data for each artist separately. Then, we train an n-gram model for each artists train set, and evaluate its perplexity on each artist's test set. By checking the artist with the lowest perplexity, we can compute the accuracy of each model. 

In [44]:
# Modify this for your particular bi-gram or tri-gram model
n = 3
smoothing = 0.1
min_freq = 2

# Run the splitting and saving process for the lyrics data
train_splits, test_splits = split_and_save_datasets(data_dir="data/lyrics/")
files = [f for f in Path("data/lyrics/").glob("*.txt") if f.name != "test_lyrics.txt"]

total = 0
correct = 0
for f in files:
  # Train a n-gram for each artists' lyrics
  train = train_splits[f.name]
  lm = NGramLanguageModel(n, train, smoothing)

  # Compute the model's perplexity on each other artist 
  perplexities = {}
  for f_2 in files:
    test = test_splits[f_2.name]
    ppl = lm.perplexity(test)
    perplexities[f_2.name] = ppl

  # Which artist has the lowest perplexity for our model
  min_perplexity_file = min(perplexities, key=perplexities.get)

  # Check whether the lowest perplexity artist is the same as which our model 
  # was trained on. 
  total += 1
  if min_perplexity_file == f.name:
    correct += 1
  
  print(f"Min perplexity for model trained on {f.name}: {min_perplexity_file}")


print(f"Accuracy score for {n}-gram model: ", 100 * (correct / total), "%")

Min perplexity for model trained on billie_eillish.txt: billie_eillish.txt
Min perplexity for model trained on down_bad.txt: so_long_london.txt
Min perplexity for model trained on ed_sheeran.txt: taylor_swift.txt
Min perplexity for model trained on green_day.txt: taylor_swift.txt
Min perplexity for model trained on so_long_london.txt: so_long_london.txt
Min perplexity for model trained on taylor_swift.txt: taylor_swift.txt
Min perplexity for model trained on the_tortured_poets_department.txt: so_long_london.txt
Accuracy score for 3-gram model:  42.857142857142854 %


### **Written 4.3.1** – Transformer Encoder Classifier Accuracy [4 points]

Report your **training accuracy** and **validation (test) accuracy** for the Transformer Encoder classifier trained in the cells above.

Run the cell below to print the final epoch's training accuracy and the overall test accuracy. Then provide a brief written summary below the output.

In [46]:
# Print training and validation accuracy
# train_acc comes from the last epoch of the training loop above
# test_accuracy comes from evaluate_model() above

print("=" * 50)
print("Transformer Encoder Classifier Results")
print("=" * 50)
print(f"Final Training Accuracy (last epoch): {train_acc:.2f}%")
print(f"Validation (Test) Accuracy:           {test_accuracy:.2f}%")

Transformer Encoder Classifier Results
Final Training Accuracy (last epoch): 85.29%
Validation (Test) Accuracy:           42.86%


### **Written 4.3.2** – Self-Attention Weight Analysis [4 points]

Analyze the self-attention weights of your trained Transformer Encoder for one of the following sequences:

- **s1**: The Tortured Poets Department  
- **s2**: So Long, London  
- **s3**: Down Bad  

Run the code below to extract the per-head attention weights for each sequence. Then answer:

1. **Which words receive the most attention** (highest average attention weight) for your chosen sequence?
2. **How does attention change across different heads?** Do different heads focus on different tokens?

In [47]:
import math

def read_sequence_from_file(filepath, min_words=3):
    """Read the first line with at least min_words words from a lyrics file."""
    lines = read_file(filepath)
    for line in lines:
        tokens = line.strip().lower().split()
        if len(tokens) >= min_words:
            return " ".join(tokens)
    return ""

# The three sequences to analyze — read from actual lyric files
SEQUENCES = {
    "s1": read_sequence_from_file("data/lyrics/the_tortured_poets_department.txt"),
    "s2": read_sequence_from_file("data/lyrics/so_long_london.txt"),
    "s3": read_sequence_from_file("data/lyrics/down_bad.txt"),
}

print("Sequences loaded:")
for k, v in SEQUENCES.items():
    print(f"  {k}: '{v}'")

def get_attention_weights(encoder_model, text, word_to_ix, max_seq_len):
    """
    Extract per-head self-attention weights from all transformer layers
    by manually stepping through each layer with need_weights=True.

    Args:
        encoder_model: trained TransformerEncoderClassifier
        text: str, the input sentence (space-separated tokens)
        word_to_ix: vocabulary dict
        max_seq_len: int

    Returns:
        tokens: list of real (non-padding) tokens
        all_attn_weights: list of tensors, one per layer,
                          each of shape (nhead, n_real, n_real)
    """
    encoder_model.eval()
    tokens = text.lower().split()
    n_real = min(len(tokens), max_seq_len)
    tokens = tokens[:n_real]

    unk_idx = word_to_ix.get("<unk>", 0)
    pad_idx = word_to_ix.get("<PAD>", 0)
    indices = [word_to_ix.get(tok, unk_idx) for tok in tokens]
    padded = indices + [pad_idx] * (max_seq_len - n_real)
    mask = [1] * n_real + [0] * (max_seq_len - n_real)

    input_tensor = torch.tensor([padded]).to(encoder_model.device)
    mask_tensor = torch.tensor([mask]).to(encoder_model.device)
    key_padding_mask = (mask_tensor == 0)  # True = padding (ignored)

    all_attn_weights = []

    with torch.no_grad():
        # Replicate the encoder's forward logic up to the transformer layers
        x = encoder_model.embedding(input_tensor) * math.sqrt(encoder_model.d_model)
        x = encoder_model.positional_encoding(x)  # (1, seq, d_model)
        x = x.transpose(0, 1)                     # (seq, 1, d_model)

        for layer in encoder_model.transformer_encoder.layers:
            # Call self-attention with per-head weights
            attn_out, attn_w = layer.self_attn(
                x, x, x,
                key_padding_mask=key_padding_mask,
                need_weights=True,
                average_attn_weights=False   # shape: (1, nhead, seq, seq)
            )
            # Store only the real-token slice: (nhead, n_real, n_real)
            all_attn_weights.append(attn_w[0, :, :n_real, :n_real].cpu())

            # Complete the rest of the layer to get the correct hidden state
            x = layer.norm1(x + layer.dropout1(attn_out))
            ff = layer.linear2(layer.dropout(layer.activation(layer.linear1(x))))
            x = layer.norm2(x + layer.dropout2(ff))

    return tokens, all_attn_weights


# Run attention analysis for all three sequences
for seq_name, seq_text in SEQUENCES.items():
    tokens, attn_weights = get_attention_weights(model, seq_text, word_to_ix, max_seq_len)
    print(f"\n{'='*60}")
    print(f"Sequence {seq_name}: '{seq_text}'")
    print(f"Tokens: {tokens}")

    for layer_idx, layer_attn in enumerate(attn_weights):
        # layer_attn shape: (nhead, n_real, n_real)
        # Average attention *received* by each token (column-wise mean across query dim)
        avg_received = layer_attn.mean(dim=0).mean(dim=0)  # avg over heads then over queries
        top_idx = avg_received.argmax().item()
        print(f"\n  Layer {layer_idx + 1}:")
        print(f"    Avg attention per token: { {t: f'{v:.3f}' for t, v in zip(tokens, avg_received.tolist())} }")
        print(f"    Token receiving most attention (avg across heads): '{tokens[top_idx]}'")

        # Per-head top token
        nhead = layer_attn.shape[0]
        head_tops = []
        for h in range(nhead):
            col_sum = layer_attn[h].mean(dim=0)  # avg attention received per token
            head_tops.append(tokens[col_sum.argmax().item()])
        print(f"    Top attended token per head: {head_tops}")

Sequences loaded:
  s1: 'i was supposed to be sent away'
  s2: 'i saw in my mind fairy lights through the mist'
  s3: 'did you really beam me up'

Sequence s1: 'i was supposed to be sent away'
Tokens: ['i', 'was', 'supposed', 'to', 'be', 'sent', 'away']

  Layer 1:
    Avg attention per token: {'i': '0.064', 'was': '0.453', 'supposed': '0.029', 'to': '0.000', 'be': '0.002', 'sent': '0.451', 'away': '0.001'}
    Token receiving most attention (avg across heads): 'was'
    Top attended token per head: ['was', 'sent']

  Layer 2:
    Avg attention per token: {'i': '0.091', 'was': '0.256', 'supposed': '0.080', 'to': '0.105', 'be': '0.081', 'sent': '0.279', 'away': '0.107'}
    Token receiving most attention (avg across heads): 'sent'
    Top attended token per head: ['was', 'sent']

Sequence s2: 'i saw in my mind fairy lights through the mist'
Tokens: ['i', 'saw', 'in', 'my', 'mind', 'fairy', 'lights', 'through', 'the', 'mist']

  Layer 1:
    Avg attention per token: {'i': '0.048', 'saw':

### **Written 4.4** – Comparison to a GPT [14 points]

Run the code provided in the cells below comparing your models to a pre-trained GPT-2 model. Use the results to answer the following questions.

1. **(2 pts)** What is the perplexity of your N-gram model? (2 points)

2. **(2 pts)** What is the perplexity of the GPT-2 model? (2 points)

3. **(6 pts)** How might you reason about the differences in perplexity between these two models? Think about **parameters**, **vocabulary size**, and **training data** of your model vs. GPT-2.

4. **(4 pts)** What are the trade-offs between a task-specific Transformer Encoder versus a large-scale generative model like GPT-2 for **classification tasks**?

The cells below compute perplexity on `data/bbc/tech-small.txt` for:
1. Your **n-gram language model** (trigram, trained on the same test corpus)
2. **GPT-2** (via the `transformers` library)

Generative pre-trained transformer (GPT) is a neural language model series created by OpenAI. The n-gram language model you trained has on average around 10K–20K parameters, compared to GPT-2's ~117 million parameters and GPT-3's ~175 billion.

You need to enable a GPU runtime from the Colab `Runtime` menu (Runtime → Change Runtime Type → Hardware Accelerator: GPU) to run the GPT-2 cell efficiently.

In [48]:
# ── N-gram perplexity on tech-small.txt ──────────────────────────────────────
tech_test = preprocess(read_file("data/bbc/tech-small.txt"), 3)
ngram_lm = NGramLanguageModel(n=3, train_data=tech_test)
ngram_ppl = ngram_lm.perplexity(tech_test)
print(f"N-gram (trigram) perplexity on tech-small.txt: {ngram_ppl:.2f}")

N-gram (trigram) perplexity on tech-small.txt: 177.48


#### Computing GPT-2's perplexity on tech-small.txt

The cell below loads **DistilGPT-2** (a smaller distilled version of GPT-2) and computes its perplexity on the same `tech-small.txt` file used for the n-gram model above. Note the variable is named `gpt2_model` to avoid overwriting the encoder `model` defined earlier.

In [49]:
# ── GPT-2 Perplexity ─────────────
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import torch

# Load DistilGPT-2 (renamed to avoid shadowing the encoder `model`)
model_id = "distilgpt2"
gpt2_model = GPT2LMHeadModel.from_pretrained(model_id)
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
gpt2_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt2_model = gpt2_model.to(gpt2_device)
gpt2_model.eval()

# Compute perplexity on tech-small.txt
with open("data/bbc/tech-small.txt", "r") as f:
    raw_text = f.read()

encodings = gpt2_tokenizer(raw_text, return_tensors="pt")
input_ids = encodings.input_ids.to(gpt2_device)

# Slide a window over the token sequence to handle long documents
max_length = gpt2_model.config.n_positions  # 1024 for GPT-2
stride = 512
nlls = []
seq_len = input_ids.size(1)

for begin_loc in range(0, seq_len, stride):
    end_loc = min(begin_loc + max_length, seq_len)
    trg_len = end_loc - begin_loc
    input_chunk = input_ids[:, begin_loc:end_loc]
    target_ids = input_chunk.clone()
    target_ids[:, :-trg_len] = -100

    with torch.no_grad():
        outputs = gpt2_model(input_chunk, labels=target_ids)
        neg_log_likelihood = outputs.loss * trg_len

    nlls.append(neg_log_likelihood)
    if end_loc == seq_len:
        break

import math
gpt2_ppl = math.exp(torch.stack(nlls).sum() / seq_len)
print(f"DistilGPT-2 perplexity on tech-small.txt: {gpt2_ppl:.2f}")

d:\Coding Projects\11611-NLP-26Spring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Coding Projects\11611-NLP-26Spring\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\17293\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate 

DistilGPT-2 perplexity on tech-small.txt: 239.23
